# 007 - Level7

## 7. Data Cleaning & Transformation

### 🧼 Limpieza y transformación de datos

- **COALESCE / IFNULL**  
  Manejan **valores faltantes** (`NULL`) y permiten asignar **valores por defecto**.

- **CAST / TRY_CAST**  
  Cambian el **tipo de dato**  
  (por ejemplo, de *string* a *integer*).

- **String functions**  
  Funciones como:
  - `TRIM`
  - `REPLACE`
  - `SUBSTRING`  
  se utilizan para **limpiar texto desordenado**.

- **Date manipulation**  
  Permite:
  - Extraer **año**, **mes** o **día**
  - Calcular **diferencias entre fechas**


In [2]:
import pandas as pd
import numpy as np
import polars as pl

In [3]:
df_matches = pd.read_csv('../Data/WorldCupMatches.csv').rename(columns=lambda col: col.lower())
df_players = pd.read_csv('../Data/WorldCupPlayers.csv').rename(columns=lambda col: col.lower())
df_cups = pd.read_csv('../Data/WorldCups.csv').rename(columns=lambda col: col.lower())

pl_matches = pl.read_csv('../Data/WorldCupMatches.csv').rename(lambda col: col.lower())
pl_players = pl.read_csv('../Data/WorldCupPlayers.csv').rename(lambda col: col.lower())
pl_cups = pl.read_csv('../Data/WorldCups.csv').rename(lambda col: col.lower())

## 🎯 Reto 7.1: Clasificación de Mundiales

### 🧠 Contexto

Trabajaremos con la tabla `worldcup.cups`.  
El objetivo es crear una **nueva columna** llamada `formato_mundial`, basada en la cantidad de **equipos clasificados** (`QualifiedTeams`).

---

### 📐 Reglas de clasificación

- **Formato Moderno**  
  Más de **24 equipos**.

- **Formato Clásico**  
  Entre **16 y 24 equipos** (inclusive).

- **Formato Antiguo**  
  Menos de **16 equipos**.


```SQL
WITH contador_equipos AS (
    SELECT
        year,
        COUNT(DISTINCT "Home Team Name") AS count_teams
    FROM worldcup.matches
    WHERE year IS NOT NULL
    GROUP BY year
)
SELECT
    year,
    count_teams,
    CASE
        WHEN count_teams > 24 THEN 'Formato Moderno'
        WHEN count_teams >= 16 THEN 'Formato Clásico' -- BETWEEN 16 AND 24 también funciona perfecto
        ELSE 'Formato Antiguo'
    END AS categoria_formato
FROM contador_equipos
ORDER BY year;
```

In [5]:
df_c = df_cups.copy()

condiciones = [
    (df_c['qualifiedteams'] > 24),
    (df_c['qualifiedteams'] >= 16)
]

opciones = ['Formato Moderno', 'Formato Clásico']

df_c['formato_tipo'] = np.select(condiciones, opciones, default='Formato Antiguo')

res = df_c[['year','qualifiedteams','formato_tipo']]

In [8]:
res = pl_cups.select([
    pl.col('year'),
    pl.col('qualifiedteams'),
    pl.when(pl.col('qualifiedteams') > 24)
    .then(pl.lit('Formato Moderno'))
    .when(pl.col('qualifiedteams') >= 16)
    .then(pl.lit('Formato Clásico'))
    .otherwise(pl.lit('Formato Antiguo'))
    .alias('formato_tipo')
])

```SQL
WITH lista_equipos AS(
    SELECT
        year,
        "Home Team Name" AS equipos
    FROM worldcup.matches
    UNION SELECT
        year,
        "Away Team Name" AS equipos
    FROM worldcup.matches
), contador_final AS(
    SELECT
        year,
        COUNT(distinct equipos) AS count_teams
    FROm lista_equipos
    WHERE year IS NOT NULL
    GROUP BY year
)
SELECT
    year,
    count_teams,
    CASE
        WHEN count_teams > 26 THEN 'Formato Moderno'
        WHEN count_teams >= 16 THEN 'Formato Clásico'
        ELSE 'Formato Antiguo'
    END AS categoria_formato
FROM contador_final
ORDER BY year;
```

In [13]:
df_m = df_matches.copy()

home = df_m[['year', 'home team name']].rename(columns={'home team name': 'team'})
away = df_m[['year', 'away team name']].rename(columns={'away team name': 'team'})

df_union = pd.concat([home,away])

df_counts = df_union.groupby('year')['team'].nunique().reset_index()
df_counts.columns = ['year', 'count_teams']

condiciones = [
    (df_counts['count_teams'] > 26),
    (df_counts['count_teams'] >= 16)
]

opciones = ['Formato Moderno', 'Formato Clásico']

df_counts['categoria_formato'] = np.select(condiciones, opciones, default='Formato Antiguo')

res = df_counts.sort_values('year')

In [17]:
res = (
    pl.concat([
        pl_matches.select(['year', pl.col('home team name').alias('team')]),
        pl_matches.select(['year', pl.col('away team name').alias('team')])
    ])
    .group_by('year')
    .agg(pl.col('team').n_unique().alias('count_teams'))
    .with_columns([
        pl.when(pl.col('count_teams') > 26).then(pl.lit('Formato Moderno'))
        .when(pl.col('count_teams') >= 16).then(pl.lit('Formato Clásico'))
        .otherwise(pl.lit('Formato Antiguo'))
        .alias('categoria_formato')
    ])
    .sort('year')
)